**descirption** In this section, feature engineering is performed to prepare data for the recommendation model. Both genre-based and tag-based representations are explored.

for:
one-hot encoding of genres  
feature-matris  
experiment with cosine similarity  
try KNN

# 1. Content-based Movie Recommender - genre

Features:
- Genres converted to text
- TF-IDF vectorization
- Cosine similarity recommendation

In [40]:
#Interactive library
import plotly.express as px

# Data handling
import numpy as np
import pandas as pd

# Machine learning
import sklearn

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.feature_extraction.text import TfidfVectorizer

# Similarity
from sklearn.metrics.pairwise import cosine_similarity as cosine_sim


In [41]:
directory_data = "../data" # Adjust the path to your data directory if necessary
movies = pd.read_csv(f"{directory_data}/raw/movies.csv")  # Adjust the path to your data directory if necessary
ratings = pd.read_csv(f"{directory_data}/raw/ratings.csv")
tags = pd.read_csv(f"{directory_data}/raw/tags.csv")

### Content-based recommendation using TF-IDF and cosine similarity

This implementation was inspired by a discussion with Johan Challita on LinkedIn, and is used as a starting point for further exploration.

The initial idea was to begin with a simple content-based similarity recommender as a baseline. In this notebook, a TF-IDF based representation is used together with cosine similarity to identify similar movies.

Rather than implementing a separate KNN model, the nearest neighbours are retrieved directly by ranking movies based on cosine similarity. This approach is equivalent to a nearest neighbour search in the feature space and provides a simple and efficient baseline for recommendation.

### TF-IDF feature representation

To transform the movie data into a numerical representation, TF-IDF vectorization is applied to the genre information.

`TfidfVectorizer` defines how text should be converted into numerical features using the TF-IDF method.

At this stage, the vectorizer is only initialized with chosen settings, such as removing English stop words. It does not yet contain any information about the dataset.

In other words, this step defines *how* the text should be processed, but no transformation has been applied yet.

The genres are first converted into a space-separated text format. This allows them to be treated as tokens and makes it possible to apply standard text processing techniques such as TF-IDF.

---

### TF-IDF transformation (fit_transform)

The `fit_transform()` method performs two steps:

- **fit**: learns the vocabulary from the dataset and calculates how important each word is across all documents  
- **transform**: converts each document into a numerical vector based on these learned values  

The result is a TF-IDF matrix where:
- each row represents a movie  
- each column represents a word (feature)  
- each value represents how important that word is for that movie  

The matrix is stored as a *sparse matrix*, meaning that only non-zero values are stored. This makes it memory efficient, since most words do not appear in most movies.

English stop words are removed during vectorization. While the MovieLens dataset is not limited to English-language movies, this approach is still applicable since genres and many tags are expressed using English terms.

However, this simplification introduces some limitations, as language variations and inconsistencies in user-generated data are not fully addressed.

In [42]:
movies["content"] = movies["genres"].str.replace("|", " ", regex=False)
# Create TF-IDF matrix
tfidf = TfidfVectorizer(stop_words="english")   #for english stop words, you can adjust this for other languages or use a custom list
tfidf_matrix = tfidf.fit_transform(movies["content"])

In [43]:

def recommend_movies_by_genres(movie_title, n=5):
    matches = movies[movies["title"].str.contains(movie_title, case=False, na=False)]

    if len(matches) == 0:
        print(f"Movie could not be found")
        return None
    if len(matches) > 1:
        print("multiple movies matched your search: ")
        print(matches[["title"]].head(10))

    
    movie_index = matches.index[0]
    title = movies.loc[movie_index, "title"]

    similarity_score = cosine_sim(tfidf_matrix[movie_index], tfidf_matrix).flatten()
       

    similar_indexes = similarity_score.argsort()[::-1]
    similar_indexes = [i for i in similar_indexes if i != movie_index]

    result = movies.iloc[similar_indexes][["title", "genres"]].head(n).copy()

    print(f"Recommendations for '{title}':")
    print(f"Movies with similarity > 0: {(similarity_score > 0).sum() - 1}")
    print(f"Movies with similarity > 0.1: {(similarity_score > 0.1).sum() - 1}")
    print(f"Movies with similarity > 0.3: {(similarity_score > 0.3).sum() - 1}")
    print(f"Movies with similarity > 0.5: {(similarity_score > 0.5).sum() - 1}")


    return result

In [44]:
recommend_by_genres("kung fu hustle", 5)

Recommendations for 'Kung Fu Hustle (Gong fu) (2004)':
Movies with similarity > 0: 30660
Movies with similarity > 0.1: 30660
Movies with similarity > 0.3: 25256
Movies with similarity > 0.5: 12866


,title,genres
64827,Take Home Pay (2019),Action|Comedy
70281,Cats & Dogs 3: Paws Unite (2020),Action|Comedy
4479,Disorganized Crime (1989),Action|Comedy
58972,Once Upon a Deadpool (2018),Action|Comedy
55743,Surge of Power: Revenge of the Sequel (2018),Action|Comedy


#Note to me: explore till KNN-solution! What differnece will it make?

The current model is based on TF-IDF representations and cosine similarity, which rely on word overlap rather than true semantic understanding. As a result, many movies receive relatively high similarity scores because they share common features (genres). This leads to a broad similarity structure where many movies are considered similar, even if they are not closely related in meaning.

# 2. Content-based Movie Recommender - genre and tags

Features:
- Genres converted to text
- TF-IDF vectorization
- combination tags and genre
- Cosine similarity recommendation

**tags per movie**

In [45]:
tags_per_movie = (
    tags.groupby("movieId")["tag"]
    .apply(lambda x: " ".join(x.dropna().astype(str)))
    .reset_index()
    .rename(columns={"tag": "tags_combined"})
)

**merge with movies**

In [46]:
movies = movies.merge(tags_per_movie, on="movieId", how="left")
movies["tags_combined"] = movies["tags_combined"].fillna("")

**Create new content**  
create new column and do not overwright the old ones.

In [47]:
movies["content_with_tags"] =  (
    movies["genres"].str.replace("|", " ", regex = False)
    + " "
    + movies["tags_combined"]
)

In [48]:
tfidf_tags = TfidfVectorizer(stop_words="english")
tfidf_matrix_tags = tfidf_tags.fit_transform(movies["content_with_tags"])


**Function för genre + tags**

In [49]:
def recommend_movies_by_genres_and_tags(movie_title, n=5):
    matches = movies[movies["title"].str.contains(movie_title, case=False, na=False)]

    if len(matches) == 0:
        print("Movie could not be found")
        return None
    
    if len(matches) > 1:
        print("Multiple movies found. Please be more specific: ")
        print(matches[["title"]].head(10))

    movie_index = matches.index[0]
    title = movies.loc[movie_index, "title"]

    similarity_score = cosine_sim(
        tfidf_matrix_tags[movie_index],
        tfidf_matrix_tags
    ).flatten()

    similar_indexes = similarity_score.argsort()[::-1]
    similar_indexes = [i for i in similar_indexes if i != movie_index]

    result = movies.iloc[similar_indexes][["title", "genres", "tags_combined"]].head(n).copy()


    print (f"Recommendations for '{title}' based on genres and tags:")  
    print(f"Movies with similarity < 0: {(similarity_score > 0).sum() - 1}")
    print(f"Movies with similarity < 0.1: {(similarity_score > 0.1).sum() - 1}")
    print(f"Movies with similarity < 0.3: {(similarity_score > 0.3).sum() - 1}")
    print(f"Movies with similarity < 0.5: {(similarity_score > 0.5).sum() - 1}")

    return result

Har jag byggt tfidf_matrix från movies["something"]?

Har jag resetat index innan?

Är movies.index fortfarande i samma ordning som när matrisen skapades?

**test function**

In [50]:
recommend_movies_by_genres_and_tags("kung fu hustle", 5)

Recommendations for 'Kung Fu Hustle (Gong fu) (2004)' based on genres and tags:
Movies with similarity < 0: 60633
Movies with similarity < 0.1: 6577
Movies with similarity < 0.3: 231
Movies with similarity < 0.5: 63


,title,genres,tags_combined
39694,The Young Vagabond (1985),Action,kung fu martial arts
30420,The Incredible Kung Fu Master (1979),Action,kung fu martial arts
50076,Mad Monkey Kung Fu (1979),Action,kung fu martial arts
35693,Ten Tigers of Kwangtung (1979),Action,kung fu martial arts
35694,The Shaolin Drunken Monk (1982),Action,kung fu martial arts


Check out differences in spelling!! 